# 01 — Asset Creation Pipeline

Converts raw PDFs from `../../data/raw_data/` into per-page assets:
- **Page images** — PNG at 200 DPI (shared across engines)
- **OCR text** — one `.txt` file per page per engine

Processing is **batch-based and resumable**: a checkpoint JSON is written
after every batch, so the notebook can be re-run without re-processing
already completed documents.

> **Recommended:** run the pipeline from the terminal via `run.py --supervise`
> rather than from this notebook, to avoid VS Code connection timeouts on large datasets.

## Supported OCR engines

| Key | Engine | Hardware |
|-----|--------|----------|
| `tesseract` | Tesseract 5 via pytesseract | CPU |
| `easyocr` | EasyOCR | GPU (CUDA) / CPU fallback |

## Setup

In [ ]:
import sys
from pathlib import Path

SRC_DIR = Path("../src/assets").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Source path:", SRC_DIR)

In [ ]:
from services.pdf_loader import PdfLoader
from services.asset_writer import AssetWriter
from services.asset_creator import AssetCreator

print("Imports OK")

## Configuration

Edit the values below, then run the remaining cells.

In [ ]:
# --- paths ---
RAW_DATA_PATH   = "../../data/raw_data"
ASSETS_OUT_PATH = "../../data/assets"

# --- engine ---
# Options: "tesseract"  |  "easyocr"
OCR_ENGINE = "tesseract"

# --- processing ---
BATCH_SIZE  = 50     # documents per checkpoint flush
LIMIT       = None   # set e.g. 10 for a quick smoke-test; None = process all
IMAGE_DPI   = 200

# --- tesseract-specific ---
TESSERACT_LANG = "eng"

# --- easyocr-specific ---
EASYOCR_LANGS = ["en"]
EASYOCR_GPU   = True   # set False if CUDA is not available

print(f"Engine: {OCR_ENGINE}")
print(f"Raw data: {Path(RAW_DATA_PATH).resolve()}")
print(f"Output:   {Path(ASSETS_OUT_PATH).resolve()}")

## Initialise OCR engine

In [ ]:
if OCR_ENGINE == "tesseract":
    from services.tesseract_ocr import TesseractOcr
    ocr = TesseractOcr(lang=TESSERACT_LANG, dpi=IMAGE_DPI)
elif OCR_ENGINE == "easyocr":
    from services.easyocr_ocr import EasyOcrEngine
    ocr = EasyOcrEngine(langs=EASYOCR_LANGS, gpu=EASYOCR_GPU)
else:
    raise ValueError(f"Unknown OCR_ENGINE: {OCR_ENGINE!r}")

print(f"OCR engine ready: {ocr.name}")

## Load documents

In [ ]:
loader    = PdfLoader(raw_data_path=RAW_DATA_PATH)
documents = loader.get_all_documents()

print(f"Found {len(documents)} valid documents")

from collections import Counter
for dtype, n in sorted(Counter(d.doc_type for d in documents).items()):
    print(f"  {dtype:<30} {n:>4}")

## Run pipeline

A checkpoint is saved to `{ASSETS_OUT_PATH}/.checkpoint-{OCR_ENGINE}.json`
after every batch. Re-running this cell skips already-completed documents.

In [ ]:
import json
from pathlib import Path
from tqdm.auto import tqdm

writer  = AssetWriter(output_base_path=ASSETS_OUT_PATH, image_dpi=IMAGE_DPI)
creator = AssetCreator(writer=writer, ocr=ocr)

docs_to_run = documents[:LIMIT] if LIMIT else documents

cp_path = Path(ASSETS_OUT_PATH) / f".checkpoint-{ocr.name}.json"
done_set = set(json.loads(cp_path.read_text())["completed"]) if cp_path.exists() else set()
pending  = [d for d in docs_to_run if f"{d.doc_type}/{d.doc_name}" not in done_set]

print(f"To process: {len(pending)}  |  Already done: {len(docs_to_run) - len(pending)}")

results = creator.create_all(documents=docs_to_run, batch_size=BATCH_SIZE, limit=LIMIT)

print("\n=== Summary ===")
print(f"  Successful : {results['successful']}")
print(f"  Failed     : {results['failed']}")
print(f"  Skipped    : {results['skipped']}")
if results['failed_docs']:
    print("  Failed docs:")
    for d in results['failed_docs'][:20]:
        print(f"    {d}")

## Sanity check

Verify that the expected files were written for the first document.

In [ ]:
sample     = documents[0]
sample_dir = Path(ASSETS_OUT_PATH) / sample.doc_type / sample.filename

print(f"Document : {sample.doc_type}/{sample.doc_name}")
print(f"Directory: {sample_dir}")
print()

if sample_dir.exists():
    for f in sorted(sample_dir.rglob("*")):
        if f.is_file():
            print(f"  {f.relative_to(sample_dir)}  ({f.stat().st_size:,} bytes)")
else:
    print("Directory not found — check ASSETS_OUT_PATH or processing errors.")